In [1]:
# ============================= DATA PREPROCESSING ==================================

In [2]:
# A) Preprocessing (video → frames + wav)
import logging
logging.getLogger().setLevel(logging.ERROR)

import os, csv, subprocess, json
from pathlib import Path

def run(cmd):
    subprocess.run(cmd, check=True)

def preprocess_video_dataset(csv_path, out_root, fps=5, sr=16000):
    """
    Reads csv with columns: emotion, filepath, split
    For each video file, extracts:
      - frames at fps into out_root/frames/<id>/
      - audio wav 16k mono into out_root/audio/<id>.wav
    Writes a manifest jsonl with resolved paths.
    """
    out_root = Path(out_root)
    (out_root/"frames").mkdir(parents=True, exist_ok=True)
    (out_root/"audio").mkdir(parents=True, exist_ok=True)
    manifest_path = out_root/"manifest.jsonl"

    with open(csv_path, "r") as f, open(manifest_path, "w") as mf:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            video_path = row["filepath"]
            emo = row["label"]
            split = row["split"]
            sample_id = f"{Path(video_path).stem}_{i}"

            frames_dir = out_root/"frames"/sample_id
            audio_path = out_root/"audio"/f"{sample_id}.wav"
            frames_dir.mkdir(parents=True, exist_ok=True)

            # Extract frames
            # -vf fps=..., scale keeps original; we'll resize in dataloader
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-vf", f"fps={fps}",
                str(frames_dir/"%05d.jpg")
            ])

            # Extract audio (mono, 16k)
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-ac", "1", "-ar", str(sr),
                str(audio_path)
            ])

            record = {
                "id": sample_id,
                "emotion": emo,
                "split": split,
                "frames_dir": str(frames_dir),
                "audio_wav": str(audio_path),
                "src_video": video_path
            }
            mf.write(json.dumps(record) + "\n")

    print("Wrote manifest:", manifest_path)

# preprocess_video_dataset(os.path.join('datasets', 'crema-d', 'updated_labels.csv'), os.path.join('datasets', 'crema-d-prep'), fps=5)
# preprocess_video_dataset(os.path.join('datasets', 'ravdess', 'updated_labels.csv'), os.path.join('datasets', 'ravdess-prep'), fps=5)

In [3]:
# B) Datasets (AffectNet image-only, AV manifests)

import json, random, math
import torch
import torchaudio
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from datasets_av import AVManifestDataset

EMO_MAP = {}  # filled after scanning labels, or define fixed mapping

def build_label_map(csv_or_manifest_paths):
    labels = set()
    for p in csv_or_manifest_paths:
        p = str(p)
        if p.endswith(".csv"):
            import csv
            with open(p,"r") as f:
                r = csv.DictReader(f)
                for row in r: labels.add(row["label"])
        else:
            with open(p,"r") as f:
                for line in f:
                    labels.add(json.loads(line)["emotion"])
    labels = sorted(list(labels))
    return {lab:i for i,lab in enumerate(labels)}

class AffectNetDataset(Dataset):
    """
    CSV columns: emotion, pth, split
    images are 96x96, we upscale for pretrained backbones.
    """
    def __init__(self, csv_path, split, label_map):
        import csv
        self.items = []
        with open(csv_path, "r") as f:
            r = csv.DictReader(f)
            for row in r:
                if row["split"] == split:
                    self.items.append(row)
        self.label_map = label_map

        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2,0.2,0.2,0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        row = self.items[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        x = self.tf(img)
        y = self.label_map[row["label"]]
        return {"image": x, "label": y}

In [4]:
# ======================================== UTILITY FUNCTIONS ==========================================

In [5]:
# C) Heterogeneity augmentations (novel evaluation + training)
import torch.nn.functional as F

import random
import torch

def corrupt_frames(frames, p=0.5):
    """
    frames: [T,3,H,W] or [B,T,3,H,W]
    Applies the SAME occlusion block to all frames (and all batch items).
    """
    if random.random() > p:
        return frames

    if frames.dim() == 4:
        T, C, H, W = frames.shape
        B = None
    elif frames.dim() == 5:
        B, T, C, H, W = frames.shape
    else:
        raise ValueError(f"frames must be 4D or 5D, got {frames.shape}")

    x0 = random.randint(0, W // 2)
    y0 = random.randint(0, H // 2)
    w  = random.randint(W // 8, W // 3)
    h  = random.randint(H // 8, H // 3)

    out = frames.clone()
    if frames.dim() == 4:
        out[:, :, y0:y0+h, x0:x0+w] = 0.0
    else:
        out[:, :, :, y0:y0+h, x0:x0+w] = 0.0

    return out


def corrupt_audio(wav, p=0.5, noise_std=0.02):
    # wav: [N]
    if random.random() > p:
        return wav
    noise = torch.randn_like(wav) * noise_std
    return torch.clamp(wav + noise, -1.0, 1.0)

def modality_dropout(frames, wav, p_drop=0.15):
    # drop one modality entirely
    r = random.random()
    if r < p_drop/2:
        frames = torch.zeros_like(frames)
    elif r < p_drop:
        wav = torch.zeros_like(wav)
    return frames, wav

def temporal_misalign(wav, max_shift=0.5, sr=16000, p=0.2):
    """
    wav: [N] or [B, N]
    randomly time-shifts waveform by up to ±max_shift seconds
    """
    if random.random() > p:
        return wav

    shift = int(random.uniform(-max_shift, max_shift) * sr)
    if shift == 0:
        return wav

    if wav.dim() == 1:
        # [N]
        if shift > 0:
            return torch.cat([torch.zeros(shift, device=wav.device), wav[:-shift]], dim=0)
        else:
            s = -shift
            return torch.cat([wav[s:], torch.zeros(s, device=wav.device)], dim=0)

    elif wav.dim() == 2:
        # [B, N]
        B, N = wav.shape
        if shift > 0:
            pad = torch.zeros(B, shift, device=wav.device, dtype=wav.dtype)
            return torch.cat([pad, wav[:, :-shift]], dim=1)
        else:
            s = -shift
            pad = torch.zeros(B, s, device=wav.device, dtype=wav.dtype)
            return torch.cat([wav[:, s:], pad], dim=1)

    else:
        raise ValueError(f"Expected wav dim 1 or 2, got {wav.dim()}")


@torch.no_grad()
def compute_classification_metrics(y_true, y_pred, num_classes):
    """
    y_true, y_pred: 1D torch tensors (CPU or GPU)
    Returns dict with:
      accuracy, macro_f1, macro_precision, macro_recall,
      per_class_precision/recall/f1, confusion_matrix
    """
    y_true = y_true.view(-1).to(torch.long)
    y_pred = y_pred.view(-1).to(torch.long)

    # Confusion matrix: rows=true, cols=pred
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long, device=y_true.device)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1

    tp = cm.diag().to(torch.float32)
    fp = cm.sum(dim=0).to(torch.float32) - tp
    fn = cm.sum(dim=1).to(torch.float32) - tp
    tn = cm.sum().to(torch.float32) - (tp + fp + fn)

    eps = 1e-8
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    accuracy = tp.sum() / (cm.sum().to(torch.float32) + eps)

    macro_precision = precision.mean()
    macro_recall    = recall.mean()
    macro_f1        = f1.mean()

    return {
        "accuracy": float(accuracy.item()),
        "macro_f1": float(macro_f1.item()),
        "macro_precision": float(macro_precision.item()),
        "macro_recall": float(macro_recall.item()),
        "per_class_precision": precision.detach().cpu().tolist(),
        "per_class_recall": recall.detach().cpu().tolist(),
        "per_class_f1": f1.detach().cpu().tolist(),
        "confusion_matrix": cm.detach().cpu(),  # tensor
    }


In [6]:
# =================================== BACKBONE MODELS + RAMER =====================================

In [7]:
# D) Model (pretrained encoders + reliability fusion

import torch
import torch.nn as nn
import torchvision

class VisionBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])  # pool output [B,512,1,1]
        self.proj = nn.Linear(512, out_dim)
        self.train_backbone = train_backbone
        # Freeze everything first
        for p in self.backbone.parameters():
            p.requires_grad = False
        
        # Unfreeze only the last ResNet block (layer4)
        for name, p in self.backbone.named_parameters():
            if "7" in name:   # in Sequential(children[:-1]), layer4 is typically index 7
                p.requires_grad = True

        # Why "7"? In ResNet18, children() indices are usually:
        # 0 conv1,1 bn1,2 relu,3 maxpool,4 layer1,5 layer2,6 layer3,7 layer4,8 avgpool,9 fc
        # Since we use children()[:-1], layer4 becomes index 7.

    def forward(self, frames):
        # frames: [B,T,3,224,224]
        B,T,C,H,W = frames.shape
        x = frames.view(B*T, C, H, W)
        z = self.backbone(x).flatten(1)  # [B*T,512]
        z = self.proj(z)                 # [B*T,out_dim]
        z = z.view(B, T, -1).mean(dim=1) # temporal average [B,out_dim]
        return z

class AudioBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_mels=128)

        base = models.mobilenet_v2(pretrained=True)
        base.classifier = nn.Identity()   # drop last layer
        for p in base.parameters():
            p.requires_grad = False

        self.backbone = base
        self.proj = nn.Linear(1280, out_dim)

    def forward(self, wav):
        spec = self.mel(wav)              # [B,128,T]
        spec = torch.log(spec + 1e-6)
        x = self.backbone(spec.unsqueeze(1).repeat(1,3,1,1))
        return self.proj(x)

class RAMER(nn.Module):
    def __init__(self, n_classes, z_dim=256):
        super().__init__()
        self.vision = VisionBackbone(out_dim=z_dim, train_backbone=False)
        self.audio  = AudioBackbone(out_dim=z_dim, train_backbone=False)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # reliability MLP: input [z_v, z_a, disagreement(4)]
        self.rel = nn.Sequential(
            nn.Linear(z_dim*2 + 4, 256),
            nn.ReLU(),
            nn.Linear(256, 2)  # r_v, r_a
        )

    @staticmethod
    def _disagreement(z_v, z_a, logits_v, logits_a):
        p_v = torch.softmax(logits_v, dim=-1)
        p_a = torch.softmax(logits_a, dim=-1)
        absdiff = torch.abs(p_v - p_a).mean(dim=-1, keepdim=True)  # [B,1]
        kl_va = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        kl_av = (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        cos   = nn.functional.cosine_similarity(z_v, z_a, dim=-1).unsqueeze(-1)  # [B,1]
        d = torch.cat([absdiff, kl_va, kl_av, cos], dim=-1)  # [B,4]
        return d

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        d = self._disagreement(z_v, z_a, logits_v, logits_a)
        r_logits = self.rel(torch.cat([z_v, z_a, d], dim=-1))
        r = torch.softmax(r_logits, dim=-1)  # [B,2]
        r_v, r_a = r[:,0:1], r[:,1:2]

        logits = r_v * logits_v + r_a * logits_a
        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "r": r,
            "d": d
        }

In [8]:
# E) Training steps (full loop with heterogeneity regularizers)

from torch.utils.data import DataLoader
import torch.optim as optim

def ramer_loss(out, y, out_corr=None, margin=0.05,
              alpha=0.3, beta=0.3, gamma=0.5, delta=0.1, ent_lambda=0.01, agree_tau=0.2):
    """
    out: forward output on clean input
    out_corr: forward output on corrupted input (optional)
    """
    logits, logits_v, logits_a = out["logits"], out["logits_v"], out["logits_a"]
    r = out["r"]  # [B,2]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha*F.cross_entropy(logits_v, y) + beta*F.cross_entropy(logits_a, y)

    # entropy regularizer (maximize entropy => minimize negative entropy)
    ent = -(r * (r.clamp_min(1e-8).log())).sum(dim=-1).mean()
    loss_ent = -ent_lambda * ent

    loss_corr = torch.tensor(0.0, device=logits.device)
    loss_agree = torch.tensor(0.0, device=logits.device)

    # agreement condition on clean
    p_v = torch.softmax(logits_v, dim=-1)
    p_a = torch.softmax(logits_a, dim=-1)
    D = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1) + \
        (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1)
    agree_mask = (D < agree_tau).float()
    # keep reliabilities near 0.5 when modalities agree
    loss_agree = (agree_mask * ((r[:,0]-0.5)**2 + (r[:,1]-0.5)**2)).mean() * delta

    # corruption-driven hinge: reliability should drop for corrupted modality
    if out_corr is not None:
        r_clean = out["r"].detach()
        r_corr  = out_corr["r"]
        # We don't know which modality got corrupted here; caller can set flags.
        # We'll assume caller provides two separate passes (corrV and corrA).
        # So this function expects out_corr for one corruption at a time.
        # We'll compute both terms using a heuristic:
        # if corruption was visual: penalize r_v increasing
        # if corruption was audio: penalize r_a increasing
        pass

    return loss_fuse + loss_uni + loss_ent + loss_agree

def train_ramer(model, dl_train, dl_val, device="cuda",
                epochs=5, lr=3e-4,
                save_path="ramer_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # decent corruption values
            frames2 = corrupt_frames(frames, p=0.20)
            wav2    = corrupt_audio(wav, p=0.20)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            # 0 corruption/clean values for benchmark
            # frames2 = corrupt_frames(frames, p=0.0)
            # wav2    = corrupt_audio(wav, p=0.0)
            # frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.0)
            # wav2    = temporal_misalign(wav2, p=0.0)

            out = model(frames2, wav2)

            out_clean = model(frames, wav)
            out_corrV = model(corrupt_frames(frames, p=1.0), wav)
            out_corrA = model(frames, corrupt_audio(wav, p=1.0))

            loss = ramer_loss(out, y)

            r0 = out_clean["r"].detach()
            rV = out_corrV["r"]
            rA = out_corrA["r"]
            loss_corrV = torch.relu(rV[:,0] - r0[:,0] + 0.05).mean()
            loss_corrA = torch.relu(rA[:,1] - r0[:,1] + 0.05).mean()
            # TODO - make loss as same as before
            loss = loss + 0.5*(loss_corrV + loss_corrA)
            # loss = loss + 0.2*(loss_corrV + loss_corrA)
            

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_ramer_metrics(model, dl_val, num_classes=model.head_v[-1].out_features, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        # save best f1-macro
        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics['accuracy'],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best RAMER model to {save_path}")
            
    return best_f1

@torch.no_grad()
def eval_ramer_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [9]:
# ============================ AFFECT-NET MODEL ====================================

In [10]:
# F) Visual pretraining on AffectNet (optional but recommended)

import torchvision.models as models

class AffectNetClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        self.m = m
    def forward(self, x): return self.m(x)

def train_affectnet(model, dl_train, dl_val, device="cuda", epochs=3, lr=2e-4,
                   save_path="affectnet_best.pt"):
    model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best = 0.0

    for ep in range(1, epochs+1):
        model.train()
        for b in dl_train:
            x = b["image"].to(device)
            y = b["label"].to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        # validation
        model.eval()
        c, t = 0, 0
        with torch.no_grad():
            for b in dl_val:
                x = b["image"].to(device)
                y = b["label"].to(device)
                pred = model(x).argmax(dim=-1)
                c += (pred == y).sum().item()
                t += y.numel()
        acc = c / max(t, 1)
        print("AffectNet epoch", ep, "val_acc", acc)

        # save best
        if acc > best:
            best = acc
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": acc
            }, save_path)
            print(f"Saved best AffectNet model to {save_path}")

    return best

In [11]:
# ============================== EARLY FUSION ===============================

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EarlyFusion(nn.Module):
    """
    Early fusion baseline:
      z_v = VisionBackbone(frames)
      z_a = AudioBackbone(wav)
      z = concat(z_v, z_a)
      logits = classifier(z)
    Returns dict with "logits" to be compatible with your eval code.
    Optionally also returns logits_v/logits_a for auxiliary loss fairness.
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # Optional unimodal heads (useful for fair auxiliary losses)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # Early fusion classifier on concatenated features
        self.fuse = nn.Sequential(
            nn.LayerNorm(z_dim * 2),
            nn.Linear(z_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes)
        )

    def forward(self, frames, wav):
        z_v = self.vision(frames)  # [B,z_dim]
        z_a = self.audio(wav)      # [B,z_dim]

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        z = torch.cat([z_v, z_a], dim=-1)
        logits = self.fuse(z)

        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a}

In [13]:
def early_fusion_loss(out, y, alpha=0.3, beta=0.3):
    loss_fuse = F.cross_entropy(out["logits"], y)
    loss_uni  = alpha * F.cross_entropy(out["logits_v"], y) + beta * F.cross_entropy(out["logits_a"], y)
    return loss_fuse + loss_uni

from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def train_early_fusion(model, dl_train, dl_val, device="cuda",
                       epochs=5, lr=3e-4,
                       save_path="earlyfusion_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.fuse[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # SAME heterogeneity pipeline as you use elsewhere
            frames2 = corrupt_frames(frames, p=0.20)
            wav2    = corrupt_audio(wav, p=0.20)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = early_fusion_loss(out, y)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_early_fusion_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best EarlyFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_early_fusion_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)


In [14]:
# ============================ LATE FUSION ====================================

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LateFusion(nn.Module):
    """
    Late fusion baseline = average of unimodal logits.
    logits = 0.5*(logits_v + logits_a)
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        logits = 0.5 * (logits_v + logits_a)

        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a}

In [16]:
def latefusion_loss(out, y, alpha=0.3, beta=0.3):
    """
    Fused CE + optional unimodal CE terms (fair + stable).
    """
    loss_fuse = F.cross_entropy(out["logits"], y)
    loss_uni  = alpha * F.cross_entropy(out["logits_v"], y) + beta * F.cross_entropy(out["logits_a"], y)
    return loss_fuse + loss_uni

from torch.utils.data import DataLoader
import torch.optim as optim

def train_latefusion(model, dl_train, dl_val, device="cuda",
                     epochs=12, lr=2e-4,
                     save_path="latefusion_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.head_v[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # same heterogeneity augmentation you use
            frames2 = corrupt_frames(frames, p=0.20)
            wav2    = corrupt_audio(wav, p=0.20)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = latefusion_loss(out, y)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_latefusion_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best LateFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_latefusion_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [17]:
# ============================ GATED FUSION ====================================

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedFusion(nn.Module):
    """
    Independent gates (not normalized to sum=1):
      g_v = sigmoid(MLP(z_v)), g_a = sigmoid(MLP(z_a))
      z = g_v*z_v + g_a*z_a
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # Gates per modality (scalar per sample)
        self.gate_v = nn.Sequential(nn.Linear(z_dim, 1), nn.Sigmoid())
        self.gate_a = nn.Sequential(nn.Linear(z_dim, 1), nn.Sigmoid())

        # Optional unimodal heads (helps stability, fair vs RAMER/ATTN)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # Fused classifier
        self.head_fuse = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        z_v = self.vision(frames)   # [B,z_dim]
        z_a = self.audio(wav)       # [B,z_dim]

        g_v = self.gate_v(z_v)      # [B,1]
        g_a = self.gate_a(z_a)      # [B,1]
        g = torch.cat([g_v, g_a], dim=1)       # [B,2]
        w = torch.softmax(g, dim=1)            # [B,2] sum-to-1
        w_v, w_a = w[:,0:1], w[:,1:2]
        z = w_v * z_v + w_a * z_a
        
        logits = self.head_fuse(z)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "g_v": g_v,
            "g_a": g_a,
            "w": w
        }

In [19]:
def gated_loss(out, y, alpha=0.3, beta=0.3, gate_reg=0.01):
    logits = out["logits"]
    logits_v = out["logits_v"]
    logits_a = out["logits_a"]
    g_v = out["g_v"]
    g_a = out["g_a"]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha*F.cross_entropy(logits_v, y) + beta*F.cross_entropy(logits_a, y)

    # Gate regularizer: mild penalty if gates saturate too hard (keeps training stable)
    # penalize distance from 0.5
    reg = ((g_v - 0.5).abs().mean() + (g_a - 0.5).abs().mean())
    loss_reg = gate_reg * reg

    return loss_fuse + loss_uni + loss_reg

from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def train_gated(model, dl_train, dl_val, device="cuda",
                epochs=12, lr=2e-4,
                save_path="gated_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    num_classes = model.head_fuse[-1].out_features

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # Same heterogeneity pipeline (you can lower p_drop for fairness if you want)
            frames2 = corrupt_frames(frames, p=0.25)
            wav2    = corrupt_audio(wav, p=0.25)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.10)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)
            loss = gated_loss(out, y, alpha=0.3, beta=0.3, gate_reg=0.01)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_gated_metrics(model, dl_val, num_classes=num_classes, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best GatedFusion model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_gated_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [20]:
# ============================ MAIN START =====================================

In [21]:
# Step 1: Build unified label map
label_map = build_label_map([
    os.path.join('datasets', 'affectNet', 'updated_labels.csv'),
    os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'),
    # "/content/prep/ravdess/manifest.jsonl",
])
print(label_map)


{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4}


In [22]:
# Step 2: DataLoaders

from torch.utils.data import DataLoader

# AffectNet
# ds_aff_tr = AffectNetDataset(os.path.join('datasets', 'affectNet', 'updated_labels.csv'), "train", label_map)
# ds_aff_va = AffectNetDataset(os.path.join('datasets', 'affectNet', 'updated_labels.csv'), "val", label_map)
# dl_aff_tr = DataLoader(ds_aff_tr, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
# dl_aff_va = DataLoader(ds_aff_va, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# CREMA-D AV
# ds_cre_tr = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "train", label_map, num_frames=8)
# ds_cre_va = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "val", label_map, num_frames=8)
# dl_cre_tr = DataLoader(ds_cre_tr, batch_size=12, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
# dl_cre_va = DataLoader(ds_cre_va, batch_size=12, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

# RAVDESS AV
# 6 speakers test
# ds_rav_te = AVManifestDataset(os.path.join('datasets', 'ravdess-prep', 'manifest.jsonl'), "test", label_map, num_frames=8)
# all speakers test
ds_rav_te = AVManifestDataset(os.path.join('datasets', 'ravdess-prep', 'manifest_all_test.jsonl'), "test", label_map, num_frames=8)
dl_rav_te = DataLoader(ds_rav_te, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

n_classes = len(label_map)

# Step 3: Train AffectNet visual classifier (optional but strong)
# aff_model = AffectNetClassifier(n_classes)
# train_affectnet(aff_model, dl_aff_tr, dl_aff_va, epochs=20)

In [23]:
%%capture
# Load AffectNet model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# aff_model = AffectNetClassifier(n_classes);
# ckpt = torch.load("affectnet_best.pt", map_location="cpu")
# aff_model.load_state_dict(ckpt["model_state"])
# aff_model.to(device).eval()

In [24]:
# Step 4: Initialize RAMER and (optionally) load visual weights
ramer = RAMER(n_classes=n_classes, z_dim=256)

# transfer resnet weights
# ramer.vision.backbone.load_state_dict(nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(), strict=True)

# Step 5: Train RAMER on CREMA-D (A/V) with heterogeneity
# best_val = train_ramer(ramer, dl_cre_tr, dl_cre_va, epochs=20, lr=1e-4)

C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [25]:
%%capture
# Load RAMER model
# ramer = RAMER(n_classes=n_classes)
# ckpt = torch.load(os.path.join("models", "ramer_best_3_epochs.pt"), map_location="cpu")
# ramer.load_state_dict(ckpt["model_state"])
# ramer.to(device).eval()

# acc_rav = eval_ramer_metrics(ramer, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS ramer audio cnn acc:", acc_rav)

In [26]:
# ==================================== EARLY FUSION MODEL ===============================

In [27]:
# early = EarlyFusion(n_classes=len(label_map), z_dim=256)
# early.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )

# # train_early_fusion(early, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="early_best.pt")

# # load early fusion model
# early = EarlyFusion(n_classes=n_classes) 
# ckpt_early = torch.load(os.path.join("models", "early_best.pt"), map_location="cpu") 
# early.load_state_dict(ckpt_early["model_state"]) 
# early.to(device).eval()

# acc_early = eval_early_fusion_metrics(early, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS early audio cnn acc:", acc_early)

In [28]:
# ================================= LATE FUSION MODEL ===================================

In [29]:
# late = LateFusion(n_classes=len(label_map), z_dim=256)
# late.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )
# # train_latefusion(late, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="latefusion_best.pt")

# # load 
# late = LateFusion(n_classes=n_classes) 
# ckpt_late = torch.load(os.path.join("models", "latefusion_best.pt"), map_location="cpu") 
# late.load_state_dict(ckpt_late["model_state"]) 
# late.to(device).eval()

# acc_late = eval_latefusion_metrics(late, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS late audio cnn acc:", acc_late)

In [30]:
# ================================= GATE FUSION MODEL ===================================

In [31]:
# gated = GatedFusion(n_classes=len(label_map), z_dim=256)
# gated.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True
# )
# # train_gated(gated, dl_cre_tr, dl_cre_va, epochs=12, lr=2e-4, save_path="gated_best.pt")

# # load
# gated = GatedFusion(n_classes=n_classes) 
# ckpt_gate = torch.load(os.path.join("models", "gated_best.pt"), map_location="cpu") 
# gated.load_state_dict(ckpt_gate["model_state"]) 
# gated.to(device).eval()

# acc_gate = eval_gated_metrics(gated, dl_rav_te, n_classes)
# print("\nZero-shot CREMA→RAVDESS gate audio cnn acc:", acc_gate)

In [32]:
# metrics_early_rav = eval_early_fusion_metrics(early, dl_rav_te, num_classes=len(label_map))
# print("EarlyFusion zero-shot on RAVDESS:", metrics_early_rav)

In [33]:
# ================================= ATTENTION MODEL =======================================

In [34]:
@torch.no_grad()
def attn_weight_stats(model, dl, device="cuda", max_batches=10):
    model.eval()
    ws = []
    for i, b in enumerate(dl):
        if i >= max_batches:
            break
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        out = model(frames, wav)
        ws.append(out["w"].detach().cpu())
    w = torch.cat(ws, 0)
    return w.mean(0).tolist(), w.std(0).tolist()

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionFusion(nn.Module):
    """
    Standard attention fusion baseline:
      z_v, z_a -> w = softmax(MLP([z_v,z_a])) -> z = wv*z_v + wa*z_a -> logits
    Also returns logits_v/logits_a for fair auxiliary supervision (optional).
    """
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()

        # Reuse your backbones if you want, or default to your existing ones
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        # modality-specific heads (optional but recommended for fair comparison)
        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # attention weights over modalities
        self.attn = nn.Sequential(
            nn.Linear(z_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 2)   # w_v, w_a
        )

        # fused classifier head
        self.head_fuse = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

    def forward(self, frames, wav):
        # frames: [B,T,3,224,224], wav: [B,N]
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        w_logits = self.attn(torch.cat([z_v, z_a], dim=-1))
        w = torch.softmax(w_logits, dim=-1)  # [B,2]
        w_v, w_a = w[:, 0:1], w[:, 1:2]

        z = w_v * z_v + w_a * z_a
        logits = self.head_fuse(z)

        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "w": w
        }

In [36]:
class LogitAttentionFusion(nn.Module):
    def __init__(self, n_classes, z_dim=256,
                 vision_backbone=None, audio_backbone=None):
        super().__init__()
        self.vision = vision_backbone if vision_backbone is not None else VisionBackbone(out_dim=z_dim)
        self.audio  = audio_backbone  if audio_backbone  is not None else AudioBackbone(out_dim=z_dim)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        self.attn = nn.Sequential(
            nn.Linear(z_dim*2, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)
        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)
        w = torch.softmax(self.attn(torch.cat([z_v, z_a], dim=-1)), dim=-1)
        logits = w[:,0:1]*logits_v + w[:,1:2]*logits_a
        return {"logits": logits, "logits_v": logits_v, "logits_a": logits_a, "w": w}

In [37]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

def attn_loss(out, y, alpha=0.3, beta=0.3, ent_lambda=0.01):
    logits, logits_v, logits_a = out["logits"], out["logits_v"], out["logits_a"]
    w = out["w"]  # [B,2]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha * F.cross_entropy(logits_v, y) + beta * F.cross_entropy(logits_a, y)

    # Encourage non-degenerate weights (optional)
    ent = -(w * (w.clamp_min(1e-8).log())).sum(dim=-1).mean()
    # loss_ent = -ent_lambda * ent
    loss_ent = -0.05 * ent   # increase from 0.01 to 0.05

    return loss_fuse + loss_uni + loss_ent

def train_attention(model, dl_train, dl_val, device="cuda",
                    epochs=5, lr=3e-4,
                    save_path="attn_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0

    # --- NEW: infer num_classes from a single forward pass ---
    model.eval()
    b0 = next(iter(dl_val))
    with torch.no_grad():
        out0 = model(b0["frames"].to(device), b0["audio"].to(device))
        num_classes = out0["logits"].shape[-1]
    # --------------------------------------------------------

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            # heavy heterogeneity
            # frames2 = corrupt_frames(frames, p=0.20)
            # wav2    = corrupt_audio(wav, p=0.20)
            # frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            # wav2    = temporal_misalign(wav2, p=0.1)

            # 0 corruption/clean values for benchmark
            frames2 = frames
            wav2    = wav

            out = model(frames2, wav2)
            loss = attn_loss(out, y, alpha=0.3, beta=0.3, ent_lambda=0.05)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_attention_metrics(model, dl_val, num_classes=num_classes, device=device)

        # NEW: optional diagnostics only if model returns "w"
        if "w" in out0:
            mean_w, std_w = attn_weight_stats(model, dl_val, device=device)
            print(
                f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
                f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f} | "
                f"attn_mean={mean_w} attn_std={std_w}"
            )
        else:
            print(
                f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
                f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
            )

        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics["accuracy"],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best Attention model to {save_path}")

    return best_f1

@torch.no_grad()
def eval_attention_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [38]:
# 2) Train AttentionFusion baseline (same backbones/settings)
# attn = AttentionFusion(n_classes=len(label_map), z_dim=256)
# attn.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True)
# train_attention(attn, dl_cre_tr, dl_cre_va, epochs=20, lr=1e-4, save_path="attn_best.pt")


# Train attention logits
# attn_logit = LogitAttentionFusion(n_classes=len(label_map), z_dim=256)
# use default weights, uncomment below line to use affect-net weights
# attn_logit.vision.backbone.load_state_dict(
#     nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(),
#     strict=True)
# train_attention(attn_logit, dl_cre_tr, dl_cre_va, epochs=20, lr=1e-4, save_path="attn_logit_best.pt")



# 3) Zero-shot eval on RAVDESS test
# m_ramer = eval_ramer_metrics(ramer, dl_rav_te, num_classes=len(label_map))
# m_attn  = eval_attention_metrics(attn, dl_rav_te, num_classes=len(label_map))

# print("RAMER  zero-shot:", m_ramer)
# print("\nATTN   zero-shot:", m_attn)

In [39]:
# Load attention models
# attn = AttentionFusion(n_classes=n_classes) 
# ckpt = torch.load(os.path.join("models", "attn_best.pt"), map_location="cpu") 
# attn.load_state_dict(ckpt["model_state"]) 
# attn.to(device).eval()

# attn_logit = LogitAttentionFusion(n_classes=n_classes) 
# ckpt_logit = torch.load(os.path.join("models", "attn_logit_best.pt"), map_location="cpu") 
# attn_logit.load_state_dict(ckpt_logit["model_state"]) 
# attn_logit.to(device).eval()

# m_attn = eval_attention_metrics(attn, dl_rav_te, num_classes=len(label_map))
# print("\nATTN zero-shot:", m_attn)

# m_attn_logit = eval_attention_metrics(attn_logit, dl_rav_te, num_classes=len(label_map))
# print("\nATTN Logits zero-shot:", m_attn_logit)

In [40]:
# #  ============================= EXPERIMENTS ZERO-SHOT TEST ==================================

# attn_logit_clean_default_weights = LogitAttentionFusion(n_classes=n_classes)
# ckpt2 = torch.load(os.path.join("mobilenet_models", "attn_logit_clean_mobilenet.pt"), map_location="cpu") 
# attn_logit_clean_default_weights.load_state_dict(ckpt2["model_state"]) 
# attn_logit_clean_default_weights.to(device).eval()

# # ------------------------------------------------------------------------------------------------------------------------

# ramer_clean_default_weights = RAMER(n_classes=n_classes)
# ckpt4 = torch.load(os.path.join("mobilenet_models", "ramer_clean_mobilenet.pt"), map_location="cpu") 
# ramer_clean_default_weights.load_state_dict(ckpt4["model_state"]) 
# ramer_clean_default_weights.to(device).eval()

# # ---------------------------------------------------------------------------------------------------------------------

# gated_clean_default_weights = GatedFusion(n_classes=n_classes)
# ckpt6 = torch.load(os.path.join("mobilenet_models", "gated_clean_mobilenet.pt"), map_location="cpu") 
# gated_clean_default_weights.load_state_dict(ckpt6["model_state"]) 
# gated_clean_default_weights.to(device).eval()

# # -------------------------------------------------------------------------------------------------------------

# late_clean_default_weights = LateFusion(n_classes=n_classes)
# ckpt8 = torch.load(os.path.join("mobilenet_models", "latefusion_clean_mobilenet.pt"), map_location="cpu") 
# late_clean_default_weights.load_state_dict(ckpt8["model_state"]) 
# late_clean_default_weights.to(device).eval()

# # -------------------------------------------------------------------------------------------------------------

# early_clean_default_weights = EarlyFusion(n_classes=n_classes)
# ckpt10 = torch.load(os.path.join("mobilenet_models", "early_clean_mobilenet.pt"), map_location="cpu") 
# early_clean_default_weights.load_state_dict(ckpt10["model_state"]) 
# early_clean_default_weights.to(device).eval()

# r1 = eval_attention_metrics(attn_logit_clean_default_weights, dl_rav_te, num_classes=len(label_map))
# print("\nAttention logit clean with default weights zero-shot:", r1)

# r2 = eval_ramer_metrics(ramer_clean_default_weights, dl_rav_te, num_classes=len(label_map))
# print("\nRAMER clean with default weights zero-shot:", r2)

# r3 = eval_early_fusion_metrics(early_clean_default_weights, dl_rav_te, num_classes=len(label_map))
# print("\nEarly clean with default weights zero-shot:", r3)

# r4 = eval_latefusion_metrics(late_clean_default_weights, dl_rav_te, num_classes=len(label_map))
# print("\nLate clean with default weights zero-shot:", r4)

# r5 = eval_gated_metrics(gated_clean_default_weights, dl_rav_te, num_classes=len(label_map))
# print("\nGate clean with default weights zero-shot:", r5)

In [41]:
# stress = stress_test_metrics(attn, dl_rav_te, num_classes=n_classes)
# for k,v in stress.items():
#     print(k, "acc", v["accuracy"], "macroF1", v["macro_f1"])

In [42]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def stress_test_metrics(model, dl, num_classes, device="cuda",
                        seeds=(42, 630, 2125),
                        noise_std=0.05,
                        modes=("clean","no_audio","no_video","audio_noise","video_occlusion","misalign")):
    """
    Returns:
      summary[m]["acc_mean"], summary[m]["acc_std"], summary[m]["f1_mean"], summary[m]["f1_std"]
    Also returns raw per-seed metrics if you want to inspect.
    """
    model.eval()

    def eval_mode_once(mode):
        all_y, all_p = [], []
        for b in dl:
            frames = b["frames"].to(device)
            wav    = b["audio"].to(device)
            y      = b["label"].to(device)

            if mode == "clean":
                pass
            elif mode == "no_audio":
                wav = torch.zeros_like(wav)
            elif mode == "no_video":
                frames = torch.zeros_like(frames)
            elif mode == "audio_noise":
                wav = corrupt_audio(wav, p=1.0, noise_std=noise_std)
            elif mode == "video_occlusion":
                frames = corrupt_frames(frames, p=1.0)
            elif mode == "misalign":
                wav = temporal_misalign(wav, p=1.0)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            out = model(frames, wav)
            pred = out["logits"].argmax(dim=-1)
            all_y.append(y.detach())
            all_p.append(pred.detach())

        y_true = torch.cat(all_y, dim=0)
        y_pred = torch.cat(all_p, dim=0)
        return compute_classification_metrics(y_true, y_pred, num_classes)

    # Run per seed
    raw = {m: [] for m in modes}
    for s in seeds:
        set_seed(int(s))
        print('processing current seed - ', s)
        for m in modes:
            raw[m].append(eval_mode_once(m))

    # Aggregate mean/std for paper tables
    summary = {}
    for m in modes:
        accs = torch.tensor([raw[m][i]["accuracy"] for i in range(len(seeds))], dtype=torch.float32)
        f1s  = torch.tensor([raw[m][i]["macro_f1"] for i in range(len(seeds))], dtype=torch.float32)
        summary[m] = {
            "acc_mean": float(accs.mean().item()),
            "acc_std":  float(accs.std(unbiased=False).item()),
            "f1_mean":  float(f1s.mean().item()),
            "f1_std":   float(f1s.std(unbiased=False).item()),
        }

    return summary, raw

# summary_ramer, raw_ramer = stress_test_metrics(ramer, dl_rav_te, num_classes=len(label_map), seeds=(42, 630, 2125))
# summary_attn,  raw_attn  = stress_test_metrics(attn,  dl_rav_te, num_classes=len(label_map), seeds=(42, 630, 2125))
# summary_ramer, raw_ramer = stress_test_metrics(ramer, dl_rav_te, num_classes=len(label_map), seeds=(42, 630))
# summary_attn,  raw_attn  = stress_test_metrics(attn,  dl_rav_te, num_classes=len(label_map), seeds=(42, 630))

# print(summary_ramer)
# print(summary_attn)

In [43]:
# ================================= EXTREME STRESS TEST ==========================================

In [44]:
import random
import torch
import torch.nn.functional as F

def audio_silence(wav):
    return torch.zeros_like(wav)

def audio_clip(wav, clip_val=0.2):
    return torch.clamp(wav, -clip_val, clip_val)

def audio_packet_loss(wav, drop_prob=0.3, chunk=400):  # chunk ~25ms at 16kHz
    # Randomly zero small chunks
    wav2 = wav.clone()
    B, N = wav2.shape
    num_chunks = max(1, N // chunk)
    mask = (torch.rand(B, num_chunks, device=wav.device) > drop_prob).float()
    mask = mask.repeat_interleave(chunk, dim=1)[:, :N]
    return wav2 * mask

def audio_add_noise_snr(wav, snr_db=0.0):
    """
    Add gaussian noise to reach target SNR in dB.
    snr_db = 0 is very harsh, -5 even harsher.
    """
    # wav: [B,N]
    eps = 1e-8
    sig_power = wav.pow(2).mean(dim=1, keepdim=True) + eps
    noise = torch.randn_like(wav)
    noise_power = noise.pow(2).mean(dim=1, keepdim=True) + eps
    snr_lin = 10 ** (snr_db / 10.0)
    scale = torch.sqrt(sig_power / (snr_lin * noise_power))
    return torch.clamp(wav + noise * scale, -1.0, 1.0)

def video_black(frames):
    return torch.zeros_like(frames)

def video_heavy_occlusion(frames, occ_ratio=0.7):
    """
    Zero out a big rectangle region on all frames.
    frames: [B,T,3,H,W]
    """
    B,T,C,H,W = frames.shape
    frames2 = frames.clone()
    occ_area = int(H * W * occ_ratio)
    # choose rectangle dims roughly
    rect_h = random.randint(int(H*0.5), H)
    rect_w = max(1, occ_area // max(rect_h,1))
    rect_w = min(rect_w, W)
    y0 = random.randint(0, max(0, H-rect_h))
    x0 = random.randint(0, max(0, W-rect_w))
    frames2[:, :, :, y0:y0+rect_h, x0:x0+rect_w] = 0.0
    return frames2

def video_downsample(frames, scale=0.25):
    """
    Severe resolution loss: downsample then upsample.
    """
    B,T,C,H,W = frames.shape
    h2 = max(1, int(H * scale))
    w2 = max(1, int(W * scale))
    x = frames.view(B*T, C, H, W)
    x = F.interpolate(x, size=(h2,w2), mode="bilinear", align_corners=False)
    x = F.interpolate(x, size=(H,W), mode="bilinear", align_corners=False)
    return x.view(B, T, C, H, W)

def video_motion_blur(frames, k=9):
    """
    Approx motion blur: average pool along width.
    """
    B,T,C,H,W = frames.shape
    x = frames.view(B*T, C, H, W)
    # blur kernel along width
    x = F.avg_pool2d(x, kernel_size=(1,k), stride=1, padding=(0,k//2))
    return x.view(B, T, C, H, W)


In [45]:
@torch.no_grad()
def eval_model_metrics_logits(model, dl, num_classes, device="cuda"):
    model.eval().to(device)
    all_y, all_p = [], []

    for b in dl:
        frames = b["frames"].to(device, non_blocking=True)
        wav    = b["audio"].to(device, non_blocking=True)
        y      = b["label"].to(device, non_blocking=True)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)


@torch.no_grad()
def extreme_stress_test(model, dl, num_classes, device="cuda"):
    """
    Extreme conditions designed to break standard fusion.
    Returns dict: condition -> metrics dict
    """
    model.eval().to(device)

    results = {}
    conditions = [
        "clean",
        "audio_silence",
        "audio_snr0",
        "audio_snr_minus5",
        "audio_packetloss_30",
        "audio_clip_0p2",
        "video_black",
        "video_occlusion_30",
        "video_occlusion_70",
        "video_downsample_25",
        "video_motionblur_9",
        "combo_audio_snr0_video_occ70",
        "combo_audio_packetloss_video_downsample",
    ]

    for cond in conditions:
        all_y, all_p = [], []
        for b in dl:
            frames = b["frames"].to(device, non_blocking=True)
            wav    = b["audio"].to(device, non_blocking=True)
            y      = b["label"].to(device, non_blocking=True)

            # Apply corruption
            if cond == "clean":
                pass
            elif cond == "audio_silence":
                wav = audio_silence(wav)
            elif cond == "audio_snr0":
                wav = audio_add_noise_snr(wav, snr_db=0.0)
            elif cond == "audio_snr_minus5":
                wav = audio_add_noise_snr(wav, snr_db=-5.0)
            elif cond == "audio_packetloss_30":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)
            elif cond == "audio_clip_0p2":
                wav = audio_clip(wav, clip_val=0.2)

            elif cond == "video_black":
                frames = video_black(frames)
            elif cond == "video_occlusion_30":
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)
            elif cond == "video_occlusion_70":
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)
            elif cond == "video_downsample_25":
                frames = video_downsample(frames, scale=0.25)
            elif cond == "video_motionblur_9":
                frames = video_motion_blur(frames, k=9)

            elif cond == "combo_audio_snr0_video_occ70":
                wav = audio_add_noise_snr(wav, snr_db=0.0)
                frames = video_heavy_occlusion(frames, occ_ratio=0.7)
            elif cond == "combo_audio_packetloss_video_downsample":
                wav = audio_packet_loss(wav, drop_prob=0.3, chunk=400)
                frames = video_downsample(frames, scale=0.25)

            out = model(frames, wav)
            pred = out["logits"].argmax(dim=-1)

            all_y.append(y.detach())
            all_p.append(pred.detach())

        y_true = torch.cat(all_y, dim=0)
        y_pred = torch.cat(all_p, dim=0)
        results[cond] = compute_classification_metrics(y_true, y_pred, num_classes)

        print(f"[{cond}] acc={results[cond]['accuracy']:.4f} macroF1={results[cond]['macro_f1']:.4f}")

    return results

In [46]:
# gated_aug_default_weights = GatedFusion(n_classes=n_classes)
# ckpt1 = torch.load(os.path.join("mobilenet_models", "gated_aug_mobilenet.pt"), map_location="cpu") 
# gated_aug_default_weights.load_state_dict(ckpt1["model_state"]) 
# gated_aug_default_weights.to(device).eval()

# gated_clean_default_weights = GatedFusion(n_classes=n_classes)
# ckpt3 = torch.load(os.path.join("mobilenet_models", "gated_clean_mobilenet.pt"), map_location="cpu") 
# gated_clean_default_weights.load_state_dict(ckpt3["model_state"]) 
# gated_clean_default_weights.to(device).eval()

# late_aug_default_weights = LateFusion(n_classes=n_classes)
# ckpt5 = torch.load(os.path.join("mobilenet_models", "latefusion_aug_mobilenet.pt"), map_location="cpu") 
# late_aug_default_weights.load_state_dict(ckpt5["model_state"]) 
# late_aug_default_weights.to(device).eval()

# late_clean_default_weights = LateFusion(n_classes=n_classes)
# ckpt7 = torch.load(os.path.join("mobilenet_models", "latefusion_clean_mobilenet.pt"), map_location="cpu") 
# late_clean_default_weights.load_state_dict(ckpt7["model_state"]) 
# late_clean_default_weights.to(device).eval()

early_aug_default_weights = EarlyFusion(n_classes=n_classes)
ckpt9 = torch.load(os.path.join("mobilenet_models", "early_aug_mobilenet.pt"), map_location="cpu") 
early_aug_default_weights.load_state_dict(ckpt9["model_state"]) 
early_aug_default_weights.to(device).eval()

early_clean_default_weights = EarlyFusion(n_classes=n_classes)
ckpt11 = torch.load(os.path.join("mobilenet_models", "early_clean_mobilenet.pt"), map_location="cpu") 
early_clean_default_weights.load_state_dict(ckpt11["model_state"]) 
early_clean_default_weights.to(device).eval()

EarlyFusion(
  (vision): VisionBackbone(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64

In [47]:
# ----------- EXTREME STRESS TEST WITH DEFAULT WEIGHTS -------------------------------

num_classes = len(label_map)

# print("=== RAMER Extreme Stress Test with clean training ===")
# ramer_ext_clean = extreme_stress_test(ramer_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== RAMER Extreme Stress Test with aug training ===")
# ramer_ext_aug = extreme_stress_test(ramer_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== ATTN LOGITS Extreme Stress Test with clean training ===")
# attn_logits_ext_clean = extreme_stress_test(attn_logit_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== ATTN LOGITS Extreme Stress Test with aug training ===")
# attn_logits_ext_aug = extreme_stress_test(attn_logit_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Gated Fusion Extreme Stress Test with clean training ===")
# gated_ext_clean = extreme_stress_test(gated_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Gated Fusion Extreme Stress Test with aug training ===")
# gated_ext_aug = extreme_stress_test(gated_aug_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Late Fusion Extreme Stress Test with clean training ===")
# late_ext_clean = extreme_stress_test(late_clean_default_weights, dl_rav_te, num_classes=num_classes)

# print("=== Late Fusion Extreme Stress Test with aug training ===")
# late_ext_aug = extreme_stress_test(late_aug_default_weights, dl_rav_te, num_classes=num_classes)

print("=== Early Fusion Extreme Stress Test with clean training ===")
early_ext_clean = extreme_stress_test(early_clean_default_weights, dl_rav_te, num_classes=num_classes)

print("=== Early Fusion Extreme Stress Test with aug training ===")
early_ext_aug = extreme_stress_test(early_aug_default_weights, dl_rav_te, num_classes=num_classes)


=== Early Fusion Extreme Stress Test with clean training ===
[clean] acc=0.3719 macroF1=0.3170
[audio_silence] acc=0.2333 macroF1=0.1250
[audio_snr0] acc=0.3344 macroF1=0.2797
[audio_snr_minus5] acc=0.3375 macroF1=0.2937
[audio_packetloss_30] acc=0.3365 macroF1=0.2789
[audio_clip_0p2] acc=0.3698 macroF1=0.3183
[video_black] acc=0.2135 macroF1=0.0924
[video_occlusion_30] acc=0.2198 macroF1=0.1075
[video_occlusion_70] acc=0.2240 macroF1=0.1165
[video_downsample_25] acc=0.2927 macroF1=0.2230
[video_motionblur_9] acc=0.2969 macroF1=0.1967
[combo_audio_snr0_video_occ70] acc=0.3208 macroF1=0.2271
[combo_audio_packetloss_video_downsample] acc=0.2500 macroF1=0.1617
=== Early Fusion Extreme Stress Test with aug training ===
[clean] acc=0.3458 macroF1=0.2574
[audio_silence] acc=0.2052 macroF1=0.0774
[audio_snr0] acc=0.3260 macroF1=0.2739
[audio_snr_minus5] acc=0.3604 macroF1=0.3091
[audio_packetloss_30] acc=0.3177 macroF1=0.2306
[audio_clip_0p2] acc=0.3469 macroF1=0.2565
[video_black] acc=0.2302

In [ ]:
# ================================= BENCHMARK ==========================================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CONDITIONS = [
    "clean",
    "audio_silence",
    "audio_snr0",
    "audio_snr_minus5",
    "audio_packetloss_30",
    "audio_clip_0p2",
    "video_black",
    "video_occlusion_70",
    "video_downsample_25",
    "video_motionblur_9",
    "combo_audio_snr0_video_occ70",
    "combo_audio_packetloss_video_downsample",
]

# Optional: group tags (for group-wise plots later)
AUDIO_CONDS = ["audio_silence","audio_snr0","audio_snr_minus5","audio_packetloss_30","audio_clip_0p2"]
VIDEO_CONDS = ["video_black","video_occlusion_70","video_downsample_25","video_motionblur_9"]
COMBO_CONDS = ["combo_audio_snr0_video_occ70","combo_audio_packetloss_video_downsample"]

In [ ]:
import pandas as pd
df = pd.read_csv('exp_results_clean.csv')

# Fill placeholders with your numbers

model_mapping = {'ramer_clean':'RAMER', 'attention_logits_clean':'ATTN_LOGITS', 'attention_baseline_clean':'ATTN_BASE'}
results = {
    "RAMER": {
        "acc": {c: None for c in CONDITIONS},
        "f1":  {c: None for c in CONDITIONS},
    },
    "ATTN_LOGITS": {
        "acc": {c: None for c in CONDITIONS},
        "f1":  {c: None for c in CONDITIONS},
    },
    "ATTN_BASE": {
        "acc": {c: None for c in CONDITIONS},
        "f1":  {c: None for c in CONDITIONS},
    },
}


# Define conditions (all columns except 'model')
CONDITIONS = [c for c in df.columns if c != "model"]

# Populate results
for _, row in df.iterrows():
    model_key = row["model"]
    
    # Skip models not in mapping
    if model_key not in model_mapping:
        continue
    
    model_name = model_mapping[model_key]
    
    for cond in CONDITIONS:
        val = row[cond]
        
        if pd.isna(val):
            continue
        
        # Split "acc / f1"
        acc_str, f1_str = val.split("/")
        acc = float(acc_str.strip())
        f1 = float(f1_str.strip())
        
        results[model_name]["acc"][cond] = acc
        results[model_name]["f1"][cond] = f1


# Example (replace with your real values)
# results["RAMER"]["acc"]["clean"] = 0.5250
# results["RAMER"]["f1"]["clean"]  = 0.4841

In [ ]:
def pareto_clean_vs_worst(results, metric="acc", title=None):
    xs, ys, labels = [], [], []
    for model, d in results.items():
        clean = d[metric]["clean"]
        worst = min(d[metric][c] for c in CONDITIONS if c != "clean")
        xs.append(clean); ys.append(worst); labels.append(model)

    plt.figure()
    plt.scatter(xs, ys)
    for x, y, lab in zip(xs, ys, labels):
        plt.text(x, y, f" {lab}", va="center")

    plt.xlabel(f"Clean {metric}")
    plt.ylabel(f"Worst-case {metric} (over corruptions)")
    plt.title(title or f"Pareto: Clean vs Worst-case ({metric})")
    plt.grid(True, alpha=0.3)
    plt.show()

pareto_clean_vs_worst(results, metric="acc")
pareto_clean_vs_worst(results, metric="f1")

In [ ]:
def robustness_auc(results, metric="acc", include_clean=False):
    out = {}
    conds = CONDITIONS if include_clean else [c for c in CONDITIONS if c != "clean"]
    for model, d in results.items():
        vals = [d[metric][c] for c in conds]
        out[model] = float(np.mean(vals))
    return out

def plot_robustness_auc(results, metric="acc", include_clean=False):
    scores = robustness_auc(results, metric=metric, include_clean=include_clean)
    models = list(scores.keys())
    vals = [scores[m] for m in models]
    plt.figure()
    plt.bar(models, vals)
    plt.ylabel(f"Mean {metric}")
    plt.title(f"Robustness AUC (mean over conditions, include_clean={include_clean})")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()
    return scores

plot_robustness_auc(results, metric="acc", include_clean=False)
plot_robustness_auc(results, metric="f1", include_clean=False)

In [ ]:
def plot_relative_drop(results, metric="acc", title=None):
    conds = [c for c in CONDITIONS if c != "clean"]
    x = np.arange(len(conds))

    plt.figure(figsize=(10,4))
    for model, d in results.items():
        clean = d[metric]["clean"]
        drops = [ (clean - d[metric][c]) / max(clean, 1e-8) for c in conds ]
        plt.plot(x, drops, marker="o", label=model)

    plt.xticks(x, conds, rotation=45, ha="right")
    plt.ylabel(f"Relative drop in {metric}")
    plt.title(title or f"Relative Degradation Profile ({metric})")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_relative_drop(results, metric="acc")
plot_relative_drop(results, metric="f1")

In [ ]:
def robustness_index(results, metric="acc", lam=0.5):
    out = {}
    robust_mean = robustness_auc(results, metric=metric, include_clean=False)
    for model, d in results.items():
        out[model] = float(lam * d[metric]["clean"] + (1-lam) * robust_mean[model])
    return out

def print_robustness_index(results, metric="acc", lam=0.5):
    out = robustness_index(results, metric=metric, lam=lam)
    for k,v in sorted(out.items(), key=lambda x: -x[1]):
        print(f"{k}: RI={v:.4f}")

print_robustness_index(results, metric="acc", lam=0.5)
print_robustness_index(results, metric="f1",  lam=0.5)
print_robustness_index(results, metric="acc", lam=0.3)  # robustness-heavy

In [ ]:
def group_means(results, metric="acc"):
    out = {}
    for model, d in results.items():
        out[model] = {
            "audio_mean": float(np.mean([d[metric][c] for c in AUDIO_CONDS])),
            "video_mean": float(np.mean([d[metric][c] for c in VIDEO_CONDS])),
            "combo_mean": float(np.mean([d[metric][c] for c in COMBO_CONDS])),
            "clean": float(d[metric]["clean"]),
        }
    return out

def plot_group_means(results, metric="acc"):
    gm = group_means(results, metric=metric)
    models = list(gm.keys())
    audio = [gm[m]["audio_mean"] for m in models]
    video = [gm[m]["video_mean"] for m in models]
    combo = [gm[m]["combo_mean"] for m in models]

    x = np.arange(len(models))
    w = 0.25

    plt.figure(figsize=(8,4))
    plt.bar(x - w, audio, width=w, label="audio")
    plt.bar(x,     video, width=w, label="video")
    plt.bar(x + w, combo, width=w, label="combo")
    plt.xticks(x, models, rotation=15)
    plt.ylabel(metric)
    plt.title(f"Group-wise robustness ({metric})")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return gm

plot_group_means(results, metric="acc")
plot_group_means(results, metric="f1")

In [ ]:
def dominance_map(results, metric="acc"):
    winners = {}
    for c in CONDITIONS:
        best_model = None
        best_val = -1e9
        for model, d in results.items():
            v = d[metric][c]
            if v > best_val:
                best_val = v
                best_model = model
        winners[c] = (best_model, best_val)
    return winners

def print_dominance(results, metric="acc"):
    winners = dominance_map(results, metric=metric)
    counts = {m: 0 for m in results.keys()}
    for c, (m, v) in winners.items():
        counts[m] += 1
        print(f"{c:30s} -> {m:12s} ({metric}={v:.4f})")
    print("\nWin counts:", counts)

print_dominance(results, metric="acc")
print_dominance(results, metric="f1")


In [ ]:
def radar_plot(results, metric="acc", models=None, title=None):
    conds = CONDITIONS
    angles = np.linspace(0, 2*np.pi, len(conds), endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(7,7))
    ax = plt.subplot(111, polar=True)

    models = models or list(results.keys())
    for model in models:
        vals = [results[model][metric][c] for c in conds]
        vals += vals[:1]
        ax.plot(angles, vals, label=model)
        ax.fill(angles, vals, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(conds, fontsize=9)
    ax.set_title(title or f"Radar: {metric} across stress conditions")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

radar_plot(results, metric="acc")
radar_plot(results, metric="f1")

In [ ]:
def plot_clean_vs_robust_mean(results, metric="acc", title=None):
    rm = robustness_auc(results, metric=metric, include_clean=False)
    plt.figure()
    for model, d in results.items():
        x = d[metric]["clean"]
        y = rm[model]
        plt.scatter([x], [y])
        plt.text(x, y, f" {model}", va="center")
    plt.xlabel(f"Clean {metric}")
    plt.ylabel(f"Robust mean {metric} (avg over corruptions)")
    plt.title(title or f"Clean vs Robust-Mean ({metric})")
    plt.grid(True, alpha=0.3)
    plt.show()

plot_clean_vs_robust_mean(results, metric="acc")
plot_clean_vs_robust_mean(results, metric="f1")

In [ ]:
Input: frames, wav
Output: fused logits
1: z_v ← VisionEncoder(frames)
2: z_a ← AudioEncoder(wav)
3: logits_v ← Head_v(z_v)
4: logits_a ← Head_a(z_a)
5: p_v ← softmax(logits_v)
6: p_a ← softmax(logits_a)
7: D1 ← mean(|p_v − p_a|)
8: D2 ← KL(p_v || p_a)
9: D3 ← KL(p_a || p_v)
10: D4 ← cosine(z_v, z_a)
11: D ← concat(D1, D2, D3, D4)
12: r ← softmax(MLP(concat(z_v, z_a, D)))
13: logits ← r_v * logits_v + r_a * logits_a
return logits